### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="labor_negotiations",
    dataset_year="1988",
    domain_str="business & marketing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5CP4Q",
    download_description="""
We download the dataset from the UCI repository and uzip it to a predefined folder.
    
mkdir -p local-data-warehouse/labor_negotiations/ && wget -P local-data-warehouse/labor_negotiations/ https://archive.ics.uci.edu/static/public/56/labor+relations.zip && unzip local-data-warehouse/labor_negotiations/labor+relations.zip -d local-data-warehouse/labor_negotiations/ && rm local-data-warehouse/labor_negotiations/labor+relations.zip 
""",
    # References
    academic_reference_bibtex="""@article{Matwin1988Collective,
  title={Collective Bargaining Review},
  author={Stan Matwin},
  year={1988},
}
""",
    academic_reference_bibtex_key="Matwin1988Collective",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
    - We merge training and test set and name columns based on provided metainformation.
    - We name the feature target "successful_settlement". 
    - We encode missing values as np.nan instead of "?".
    - Anomaly: "standby_pay" and "wage_increase_third_year" features contain only 9 and 15 (out of 57) non-nan values, respectively.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="successful_settlement",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="successful_settlement",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

column_names = [
"duration_of_agreement",
"wage_increase_first_year",
"wage_increase_second_year",
"wage_increase_third_year",
"cost_of_living_adjustment",
"working_hours",
"pension_contribution",
"standby_pay",
"supplement_on_shift",
"education_allowance",
"holidays_days",
"paid_vacation",
"longterm_disability_assistance",
"dental_plan_contribution",
"bereavement_support",
"health_plan_contribution",
"successful_settlement",
]

df_train = pd.read_csv(dataset_mold.path / "C4.5/labor-neg.data", header=None, index_col=False, names=column_names)
df_test = pd.read_csv(dataset_mold.path / "C4.5/labor-neg.test", header=None, index_col=False, names=column_names)
df = pd.concat([df_train, df_test], ignore_index=True)
df.replace("?", np.nan, inplace=True)

cat_columns = ["successful_settlement", "cost_of_living_adjustment", "pension_contribution", "education_allowance", "paid_vacation", "longterm_disability_assistance", "dental_plan_contribution", "bereavement_support", "health_plan_contribution"]
df[cat_columns] = df[cat_columns].astype("category")

non_cat_columns = df.columns[~df.dtypes.eq("category")]
df[non_cat_columns] = df[non_cat_columns].apply(pd.to_numeric, errors="coerce")
df["successful_settlement"] = df["successful_settlement"].map({"good": "Yes", "bad": "No"})

## Data Checks

In [67]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 57
Columns: 17
Use sampling: False (sample size: 57)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['wage_increase_first_year', 'wage_increase_second_year', 'supplement_on_shift', 'wage_increase_third_year', 'working_hours', 'standby_pay', 'holidays_days', 'duration_of_agreement', 'cost_of_living_adjustment', 'dental_plan_contribution']
Rows remaining as candidates after top-10 filter: 0 (of 57)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [68]:
# Sample Rows
df_head

,duration_of_agreement,wage_increase_first_year,wage_increase_second_year,wage_increase_third_year,cost_of_living_adjustment,working_hours,pension_contribution,standby_pay,supplement_on_shift,education_allowance,holidays_days,paid_vacation,longterm_disability_assistance,dental_plan_contribution,bereavement_support,health_plan_contribution,successful_settlement
0,1.0,5.0,NaN,NaN,NaN,40.0,NaN,NaN,2.0,NaN,11.0,average,NaN,NaN,yes,NaN,Yes
1,2.0,4.5,5.8,NaN,NaN,35.0,ret_allw,NaN,NaN,yes,11.0,below average,NaN,full,NaN,full,Yes
2,NaN,NaN,NaN,NaN,NaN,38.0,empl_contr,NaN,5.0,NaN,11.0,generous,yes,half,yes,half,Yes
3,3.0,3.7,4.0,5.0,tc,NaN,NaN,NaN,NaN,yes,NaN,NaN,NaN,NaN,yes,NaN,Yes
4,3.0,4.5,4.5,5.0,NaN,40.0,NaN,NaN,NaN,NaN,12.0,average,NaN,half,yes,half,Yes


In [69]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,education_allowance,category,35.0,61.40,2.0,"no, yes"
1,pension_contribution,category,30.0,52.63,3.0,"empl_contr, none, ret_allw"
2,longterm_disability_assistance,category,29.0,50.88,2.0,"yes, no"
3,bereavement_support,category,27.0,47.37,2.0,"yes, no"
4,cost_of_living_adjustment,category,20.0,35.09,3.0,"none, tcf, tc"
5,dental_plan_contribution,category,20.0,35.09,3.0,"half, full, none"
6,health_plan_contribution,category,20.0,35.09,3.0,"full, half, none"
7,paid_vacation,category,6.0,10.53,3.0,"below average, average, generous"
8,successful_settlement,category,0.0,0.00,2.0,"Yes, No"
9,standby_pay,float64,48.0,84.21,7.0,"2.0, 12.0, 13.0, 8.0, 4.0, 14.0, 10.0"


In [70]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
duration_of_agreement,56.0,2.160714,0.707795,1.0,3.0
wage_increase_first_year,56.0,3.803571,1.370596,2.0,7.0
wage_increase_second_year,46.0,3.971739,1.164028,2.0,7.0
wage_increase_third_year,15.0,3.913333,1.304315,2.0,5.1
working_hours,51.0,38.039216,2.505680,27.0,40.0
standby_pay,9.0,7.444444,5.027701,2.0,14.0
supplement_on_shift,31.0,4.870968,4.544168,0.0,25.0
holidays_days,53.0,11.094340,1.259795,9.0,15.0


In [71]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                         rank                             
bereavement_support            1               yes     27  47.37
                               2              <NA>     27  47.37
                               3                no      3   5.26
cost_of_living_adjustment      1              none     22  38.60
                               2              <NA>     20  35.09
                               3               tcf      8  14.04
                               4                tc      7  12.28
dental_plan_contribution       1              <NA>     20  35.09
                               2              half     15  26.32
                               3              full     13  22.81
                               4              none      9  15.79
education_allowance            1              <NA>     35  61.40
                               2                no     12  21.05
                               3               yes     10  17.54
health_plan_contribution       1              full     20  35.09
                               2              <NA>     20  35.09
                               3              half      9  15.79
                               4              none      8  14.04
longterm_disability_assistance 1              <NA>     29  50.88
                               2               yes     20  35.09
                               3                no      8  14.04
paid_vacation                  1     below average     18  31.58
                               2           average     17  29.82
                               3          generous     16  28.07
                               4              <NA>      6  10.53
pension_contribution           1              <NA>     30  52.63
                               2        empl_contr     12  21.05
                               3              none     11  19.30
                               4          ret_allw      4   7.02
successful_settlement          1               Yes     37  64.91
                               2                No     20  35.09

In [72]:
# Target Distribution
target_df

,count,pct
successful_settlement,,
Yes,37,64.91
No,20,35.09


## Task Curation

In [73]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [74]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [75]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d05b7-3fc3-7372-993b-14a974f1def8
38497a4857f7e8b5a4558d2b510b02364853201593d64de7edfc4a0f6473d506
